In [1]:
import asyncio
import re
from typing import List, Dict
from google.oauth2 import service_account
from googleapiclient.discovery import build
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [ ]:

RAW_SHEET_NAME = "Sheet1" 
PROCESSED_SHEET_NAME = "Sheet2"  
CONTENT_DELIMITER = "--- NEXT CONTENT FROM HERE ---"
CATEGORIES = ["ABOUT_US", "EBOOK", "COURSES", "RECENT_BLOG", "TESTIMONIALS", "WEBINAR", "SERVICES", "PODCAST", "SHOP"]
COLUMN_TO_WRITE_URL_TO = {
    "ABOUT_US": "M", "EBOOK": "N", "COURSES": "O", "RECENT_BLOG": "P",
    "TESTIMONIALS": "Q", "WEBINAR": "R", "SERVICES": "S", "PODCAST": "T", "SHOP": "U"
}
EXTRACTION_METADATA_COLUMN = "V"
CATEGORY_THRESHOLDS = {
    "ABOUT_US": 200, "EBOOK": 200, "COURSES": 300, "RECENT_BLOG": 450, 
    "TESTIMONIALS": 100, "WEBINAR": 150, "SERVICES": 150, "PODCAST": 200, "SHOP": 100
}
OPENAI_API_KEY = ""
SHEET_URL = "https://docs.google.com/spreadsheets/d/1wDaFAe5ayIB8zSyjub9QQfsFtyQcHdbgOR4YZmeY8vQ/edit?usp=sharing"
CREDENTIALS_FILE = "data/url-to-email-445616-cebe4868914f.json"  
PROMPT_TEMPLATES = {
    "RECENT_BLOG": """
You are provided with multiple blog posts. Your task is to evaluate each post based on the following criteria and select the one with the highest total score. Then, output **only** the content text of the selected blog post. Do not include any explanations, scores, rankings, metadata, or additional information.

**Evaluation Criteria:**
1. *Recency*: Score based on how recently the post was published, using the publication date provided.
2. *Relevance to Prospect's Industry*: Score based on how relevant the content is to the specified industry or keywords, using the excerpt provided.
3. *Post Length*: Score based on the word count, with a minimum of 200 words.

**Scoring Guidelines**:
- *Recency*:
  - Within the last month: 10
  - 1-3 months ago: 8
  - 3-6 months ago: 6
  - 6-12 months ago: 4
  - Over 12 months ago: 2
- *Relevance to Prospect's Industry*:
  - Highly relevant (multiple keywords or strong topic alignment): 10
  - Moderately relevant (some keywords or partial alignment): 7
  - Slightly relevant (few keywords or weak alignment): 5
  - Not relevant: 0
- *Post Length*:
  - Less than 200 words: 0
  - 200-500 words: 5
  - 500-1000 words: 7
  - 1000-2000 words: 9
  - 2000+ words: 10

**Final Output**:
- Output **only** the content text of the blog post with the highest total score.
- Do **not** include any explanations, scores, rankings, metadata (e.g., "Content length", "Line count", "Word Count"), or any other information.
- Do **not** mention other posts or provide any additional context.

{content}
"""
}

In [3]:
# # GoogleSheetsManager class for API interactions
# class GoogleSheetsManager:
#     def __init__(self, credentials_file):
#         scopes = ['https://www.googleapis.com/auth/spreadsheets']
#         creds = service_account.Credentials.from_service_account_file(credentials_file, scopes=scopes)
#         self.service = build('sheets', 'v4', credentials=creds)

#     def extract_spreadsheet_id(self, sheet_url):
#         pattern = r'/spreadsheets/d/([a-zA-Z0-9-_]+)'
#         match = re.search(pattern, sheet_url)
#         if match:
#             return match.group(1)
#         raise ValueError(f"Invalid Google Sheet URL: {sheet_url}")

# # Parse content into pieces based on delimiter and metadata
# def parse_content_pieces(content, expected_count, category, row_num):
#     if not content or not content.strip():
#         return []
#     cleaned_content = content.strip()
#     delimiters_to_try = [
#         CONTENT_DELIMITER,
#         CONTENT_DELIMITER.strip(),
#         "-----NEXT CONTENT FROM HERE-----",
#         "--- --NEXT CONTENT FROM HERE-----",
#         "-----NEXT CONTENT FROM HERE--- --",
#     ]
#     pieces = None
#     delimiter_used = None
#     for delimiter in delimiters_to_try:
#         if delimiter in cleaned_content:
#             pieces = cleaned_content.split(delimiter)
#             delimiter_used = delimiter
#             break
#     if pieces is None:
#         pieces = [cleaned_content]
#     else:
#         pass
#     cleaned_pieces = [piece.strip() for piece in pieces if piece.strip()]
#     if expected_count == 1 and len(cleaned_pieces) == 1:
#         return cleaned_pieces
#     elif expected_count > 1:
#         if len(cleaned_pieces) >= expected_count:
#             return cleaned_pieces[:expected_count]
#         else:
#             return cleaned_pieces
#     else:
#         return cleaned_pieces

# # Read and organize data from the input sheet
# async def read_data_from_sheet(spreadsheet_id, sheet_mgr, categories):
#     category_columns = {cat: COLUMN_TO_WRITE_URL_TO.get(cat.upper()) for cat in categories}
#     if any(col is None for col in category_columns.values()):
#         raise ValueError("One or more categories do not have a defined column in COLUMN_TO_WRITE_URL_TO")
#     metadata_column = EXTRACTION_METADATA_COLUMN
#     ranges = [f"{RAW_SHEET_NAME}!{col}2:{col}" for col in category_columns.values()] + [f"{RAW_SHEET_NAME}!{metadata_column}2:{metadata_column}"]
#     try:
#         response = await asyncio.to_thread(
#             sheet_mgr.service.spreadsheets().values().batchGet(spreadsheetId=spreadsheet_id, ranges=ranges).execute
#         )
#     except Exception as e:
#         raise RuntimeError(f"Failed to read data from spreadsheet {spreadsheet_id}: {str(e)}")
#     value_ranges = response.get('valueRanges', [])
#     column_data = {}
#     for i, col in enumerate(category_columns.values()):
#         column_data[col] = value_ranges[i].get('values', [])
#     column_data[metadata_column] = value_ranges[-1].get('values', [])
#     num_rows = max(len(values) for values in column_data.values()) if column_data else 0
#     data = []
#     for i in range(num_rows):
#         row_data = {}
#         metadata_value = column_data[metadata_column][i][0] if i < len(column_data[metadata_column]) and column_data[metadata_column][i] else ''
#         category_n_dict = {}
#         for part in metadata_value.split(','):
#             if '=' in part:
#                 cat, n_str = part.split('=', 1)
#                 try:
#                     n = int(n_str.strip())
#                     category_n_dict[cat.strip().upper()] = n
#                 except ValueError:
#                     pass
#         for category in categories:
#             category_upper = category.upper()
#             if category_upper in category_n_dict and category_n_dict[category_upper] > 0:
#                 col = category_columns[category]
#                 content = column_data[col][i][0] if i < len(column_data[col]) and column_data[col][i] else ''
#                 expected_count = category_n_dict[category_upper]
#                 parsed_pieces = parse_content_pieces(content, expected_count, category, i+2)
#                 if parsed_pieces:
#                     row_data[category] = parsed_pieces
#         data.append(row_data)
#     return data

# # Format RECENT_BLOG content for AI processing
# def explore_all_content1(data, row_number, category):
#     try:
#         row_data = data[row_number - 2]
#         if category in row_data:
#             pieces = row_data[category]
#             output = []
#             for idx, content in enumerate(pieces, start=1):
#                 output.append(f"*Blog Post {idx}*:")
#                 output.append(content)
#                 output.append("")
#                 output.append(f"Content length: {len(content)} characters")
#                 output.append(f"Line count: {len(content.splitlines())}")
#                 output.append(f"Word Count: {len(content.split())}")
#             return "\n".join(output)
#         else:
#             return f"Error: Category '{category}' not found in row {row_number}."
#     except IndexError:
#         return f"Error: Row {row_number} not found. Available rows: 2 to {len(data) + 1}"

# # Set up LangChain pipeline for AI processing
# def create_chain(category):
#     if category not in PROMPT_TEMPLATES:
#         raise ValueError(f"No prompt defined for category {category}")
#     prompt = PromptTemplate(
#         input_variables=["content"],
#         template=PROMPT_TEMPLATES[category]
#     )
#     llm = ChatOpenAI(model="gpt-4", api_key=OPENAI_API_KEY)
#     return prompt | llm | StrOutputParser()

# # Process RECENT_BLOG row with AI
# async def process_row_with_langchain(content_output, category):
#     chain = create_chain(category)
#     if not content_output.strip() or "Error:" in content_output:
#         return f"No valid content provided for {category}"
#     try:
#         response = await chain.ainvoke({"content": content_output})
#         return response
#     except Exception as e:
#         return f"Error processing row for {category}"

# # Process a category column according to the algorithm
# async def process_category(data, category, sheet_mgr, spreadsheet_id):
#     column = COLUMN_TO_WRITE_URL_TO[category.upper()]
#     values = []
#     for row_idx in range(2, len(data) + 2):
#         row_data = data[row_idx - 2]
#         pieces = row_data.get(category, [])
#         if not pieces:
#             output = "no content"
#         elif len(pieces) == 1:
#             output = pieces[0]
#         else:
#             if category == "RECENT_BLOG":
#                 content_output = explore_all_content1(data, row_idx, category)
#                 if "Error:" in content_output:
#                     output = "No valid content provided"
#                 else:
#                     output = await process_row_with_langchain(content_output, category)
#             else:
#                 threshold = CATEGORY_THRESHOLDS.get(category.upper(), 0)
#                 candidates = [p for p in pieces if len(p.split()) >= threshold]
#                 if candidates:
#                     output = max(candidates, key=lambda p: len(p.split()))
#                 else:
#                     output = max(pieces, key=lambda p: len(p.split()))
#         values.append([output])
#     range_name = f"{PROCESSED_SHEET_NAME}!{column}2:{column}{len(data) + 1}"
#     try:
#         sheet_mgr.service.spreadsheets().values().update(
#             spreadsheetId=spreadsheet_id,
#             range=range_name,
#             valueInputOption='RAW',
#             body={'values': values}
#         ).execute()
#     except Exception as e:
#         pass

# # Main function to orchestrate the pipeline
# async def main():
#     try:
#         sheet_mgr = GoogleSheetsManager(CREDENTIALS_FILE)
#         spreadsheet_id = sheet_mgr.extract_spreadsheet_id(SHEET_URL)
#         data = await read_data_from_sheet(spreadsheet_id, sheet_mgr, CATEGORIES)
#         for category in CATEGORIES:
#             await process_category(data, category, sheet_mgr, spreadsheet_id)
#     except Exception as e:
#         pass

# # Run the pipeline
# # if __name__ == "__main__":
# #     asyncio.run(main())

# await main()

In [4]:
# GoogleSheetsManager class for API interactions
class GoogleSheetsManager:
    def __init__(self, credentials_file):
        print(f"🔧 Initializing GoogleSheetsManager with credentials: {credentials_file}")
        scopes = ['https://www.googleapis.com/auth/spreadsheets']
        creds = service_account.Credentials.from_service_account_file(credentials_file, scopes=scopes)
        self.service = build('sheets', 'v4', credentials=creds)
        print("✅ Google Sheets service initialized successfully")

    def extract_spreadsheet_id(self, sheet_url):
        print(f"🔍 Extracting spreadsheet ID from URL: {sheet_url}")
        pattern = r'/spreadsheets/d/([a-zA-Z0-9-_]+)'
        match = re.search(pattern, sheet_url)
        if match:
            spreadsheet_id = match.group(1)
            print(f"✅ Extracted spreadsheet ID: {spreadsheet_id}")
            return spreadsheet_id
        raise ValueError(f"Invalid Google Sheet URL: {sheet_url}")

# Parse content into pieces based on delimiter and metadata
def parse_content_pieces(content, expected_count, category, row_num):
    print(f"\n📝 Parsing content for {category} (Row {row_num})")
    print(f"   Expected count: {expected_count}")
    print(f"   Content length: {len(content) if content else 0} characters")
    
    if not content or not content.strip():
        print("   ⚠️  No content to parse")
        return []
    
    cleaned_content = content.strip()
    delimiters_to_try = [
        CONTENT_DELIMITER,
        CONTENT_DELIMITER.strip(),
        "-----NEXT CONTENT FROM HERE-----",
        "--- NEXT CONTENT FROM HERE ---"
        "--- --NEXT CONTENT FROM HERE-----",
        "-----NEXT CONTENT FROM HERE--- --",
    ]
    
    pieces = None
    delimiter_used = None
    
    print(f"   🔍 Trying {len(delimiters_to_try)} different delimiters...")
    for i, delimiter in enumerate(delimiters_to_try):
        if delimiter in cleaned_content:
            pieces = cleaned_content.split(delimiter)
            delimiter_used = delimiter
            print(f"   ✅ Found delimiter #{i+1}: '{delimiter[:30]}...'")
            break
    
    if pieces is None:
        print("   ℹ️  No delimiter found, treating as single piece")
        pieces = [cleaned_content]
    else:
        print(f"   📊 Split into {len(pieces)} raw pieces")
    
    cleaned_pieces = [piece.strip() for piece in pieces if piece.strip()]
    print(f"   🧹 After cleaning: {len(cleaned_pieces)} valid pieces")
    
    for i, piece in enumerate(cleaned_pieces):
        print(f"      Piece {i+1}: {len(piece)} chars, {len(piece.split())} words")
    
    if expected_count == 1 and len(cleaned_pieces) == 1:
        print("   ✅ Expected 1, got 1 - returning as is")
        return cleaned_pieces
    elif expected_count > 1:
        if len(cleaned_pieces) >= expected_count:
            result = cleaned_pieces[:expected_count]
            print(f"   ✂️  Taking first {expected_count} pieces")
            return result
        else:
            print(f"   ⚠️  Expected {expected_count}, only got {len(cleaned_pieces)}")
            return cleaned_pieces
    else:
        print(f"   ↩️  Returning all {len(cleaned_pieces)} pieces")
        return cleaned_pieces

# Read and organize data from the input sheet
async def read_data_from_sheet(spreadsheet_id, sheet_mgr, categories):
    print(f"\n📖 Reading data from spreadsheet: {spreadsheet_id}")
    print(f"   Categories to process: {categories}")
    
    category_columns = {cat: COLUMN_TO_WRITE_URL_TO.get(cat.upper()) for cat in categories}
    print(f"   Column mapping: {category_columns}")
    
    if any(col is None for col in category_columns.values()):
        missing = [cat for cat, col in category_columns.items() if col is None]
        print(f"   ❌ Missing columns for categories: {missing}")
        raise ValueError("One or more categories do not have a defined column in COLUMN_TO_WRITE_URL_TO")
    
    metadata_column = EXTRACTION_METADATA_COLUMN
    ranges = [f"{RAW_SHEET_NAME}!{col}2:{col}" for col in category_columns.values()] + [f"{RAW_SHEET_NAME}!{metadata_column}2:{metadata_column}"]
    print(f"   📋 Reading ranges: {ranges}")
    
    try:
        response = await asyncio.to_thread(
            sheet_mgr.service.spreadsheets().values().batchGet(spreadsheetId=spreadsheet_id, ranges=ranges).execute
        )
        print("   ✅ Successfully read data from Google Sheets")
    except Exception as e:
        print(f"   ❌ Failed to read data: {str(e)}")
        raise RuntimeError(f"Failed to read data from spreadsheet {spreadsheet_id}: {str(e)}")
    
    value_ranges = response.get('valueRanges', [])
    print(f"   📊 Got {len(value_ranges)} value ranges")
    
    column_data = {}
    for i, col in enumerate(category_columns.values()):
        column_data[col] = value_ranges[i].get('values', [])
        print(f"   Column {col}: {len(column_data[col])} rows")
    
    column_data[metadata_column] = value_ranges[-1].get('values', [])
    print(f"   Metadata column {metadata_column}: {len(column_data[metadata_column])} rows")
    
    num_rows = max(len(values) for values in column_data.values()) if column_data else 0
    print(f"   📏 Total rows to process: {num_rows}")
    
    data = []
    for i in range(num_rows):
        print(f"\n   🔍 Processing row {i+2}...")
        row_data = {}
        
        # Get metadata
        metadata_value = column_data[metadata_column][i][0] if i < len(column_data[metadata_column]) and column_data[metadata_column][i] else ''
        print(f"      Metadata: '{metadata_value}'")
        
        # Parse category counts
        category_n_dict = {}
        for part in metadata_value.split(','):
            if '=' in part:
                cat, n_str = part.split('=', 1)
                try:
                    n = int(n_str.strip())
                    category_n_dict[cat.strip().upper()] = n
                    print(f"      Found: {cat.strip().upper()} = {n}")
                except ValueError:
                    print(f"      ⚠️  Invalid count for: {part}")
                    pass
        
        print(f"      Category counts: {category_n_dict}")
        
        # Process each category
        for category in categories:
            category_upper = category.upper()
            print(f"      🔍 Checking category: {category_upper}")
            
            if category_upper in category_n_dict and category_n_dict[category_upper] > 0:
                print(f"         ✅ Count > 0, processing...")
                col = category_columns[category]
                content = column_data[col][i][0] if i < len(column_data[col]) and column_data[col][i] else ''
                expected_count = category_n_dict[category_upper]
                
                print(f"         Content preview: '{content[:100]}...' ({len(content)} chars)")
                
                parsed_pieces = parse_content_pieces(content, expected_count, category, i+2)
                if parsed_pieces:
                    row_data[category] = parsed_pieces
                    print(f"         ✅ Added {len(parsed_pieces)} pieces to row data")
                else:
                    print(f"         ⚠️  No valid pieces found")
            else:
                print(f"         ❌ Skipping (count = {category_n_dict.get(category_upper, 0)})")
        
        data.append(row_data)
        print(f"      Row {i+2} final data: {list(row_data.keys())}")
    
    print(f"✅ Data reading complete. Processed {len(data)} rows")
    return data

# Format RECENT_BLOG content for AI processing
def explore_all_content1(data, row_number, category):
    print(f"\n🤖 Formatting content for AI processing...")
    print(f"   Row: {row_number}, Category: {category}")
    
    try:
        row_data = data[row_number - 2]
        print(f"   Row data keys: {list(row_data.keys())}")
        
        if category in row_data:
            pieces = row_data[category]
            print(f"   Found {len(pieces)} pieces to format")
            
            output = []
            for idx, content in enumerate(pieces, start=1):
                print(f"   Formatting piece {idx}: {len(content)} chars")
                output.append(f"*Blog Post {idx}*:")
                output.append(content)
                output.append("")
                output.append(f"Content length: {len(content)} characters")
                output.append(f"Line count: {len(content.splitlines())}")
                output.append(f"Word Count: {len(content.split())}")
            
            formatted_output = "\n".join(output)
            print(f"   ✅ Formatted output: {len(formatted_output)} total chars")
            return formatted_output
        else:
            error_msg = f"Error: Category '{category}' not found in row {row_number}."
            print(f"   ❌ {error_msg}")
            return error_msg
    except IndexError:
        error_msg = f"Error: Row {row_number} not found. Available rows: 2 to {len(data) + 1}"
        print(f"   ❌ {error_msg}")
        return error_msg

# Set up LangChain pipeline for AI processing
def create_chain(category):
    print(f"🔗 Creating LangChain pipeline for category: {category}")
    
    if category not in PROMPT_TEMPLATES:
        error_msg = f"No prompt defined for category {category}"
        print(f"   ❌ {error_msg}")
        raise ValueError(error_msg)
    
    prompt = PromptTemplate(
        input_variables=["content"],
        template=PROMPT_TEMPLATES[category]
    )
    
    print(f"   📝 Using prompt template for {category}")
    print(f"   🤖 Initializing ChatOpenAI with GPT-4")
    
    llm = ChatOpenAI(model="gpt-4", api_key=OPENAI_API_KEY)
    chain = prompt | llm | StrOutputParser()
    
    print(f"   ✅ Chain created successfully")
    return chain

# Process RECENT_BLOG row with AI
async def process_row_with_langchain(content_output, category):
    print(f"\n🚀 Processing with AI (LangChain)...")
    print(f"   Category: {category}")
    print(f"   Input length: {len(content_output)} characters")
    
    chain = create_chain(category)
    
    if not content_output.strip() or "Error:" in content_output:
        error_msg = f"No valid content provided for {category}"
        print(f"   ❌ {error_msg}")
        return error_msg
    
    try:
        print("   🔄 Sending to AI...")
        response = await chain.ainvoke({"content": content_output})
        print(f"   ✅ AI processing successful!")
        print(f"   📤 Response length: {len(response)} characters")
        print(f"   📤 Response preview: '{response[:200]}...'")
        return response
    except Exception as e:
        error_msg = f"Error processing row for {category}: {str(e)}"
        print(f"   ❌ {error_msg}")
        return f"Error processing row for {category}"

# Process a category column according to the algorithm
async def process_category(data, category, sheet_mgr, spreadsheet_id):
    print(f"\n🏭 Processing category: {category}")
    
    column = COLUMN_TO_WRITE_URL_TO[category.upper()]
    print(f"   Target column: {column}")
    
    values = []
    
    for row_idx in range(2, len(data) + 2):
        print(f"\n   📋 Processing row {row_idx}...")
        row_data = data[row_idx - 2]
        pieces = row_data.get(category, [])
        
        print(f"      Found {len(pieces)} pieces for {category}")
        
        if not pieces:
            output = "no content"
            print(f"      ❌ No pieces found -> '{output}'")
        elif len(pieces) == 1:
            output = pieces[0]
            print(f"      📋 Single piece -> copying directly ({len(output)} chars)")
        else:
            print(f"      🔄 Multiple pieces ({len(pieces)}) - applying processing logic...")
            
            if category == "RECENT_BLOG":
                print(f"         🤖 RECENT_BLOG category -> AI processing")
                content_output = explore_all_content1(data, row_idx, category)
                
                if "Error:" in content_output:
                    output = "No valid content provided"
                    print(f"         ❌ Error in content formatting -> '{output}'")
                else:
                    print(f"         🚀 Sending to AI processing...")
                    output = await process_row_with_langchain(content_output, category)
                    print(f"         ✅ AI processing complete")
            else:
                print(f"         📏 Non-blog category -> length-based selection")
                threshold = CATEGORY_THRESHOLDS.get(category.upper(), 0)
                print(f"         📊 Word threshold: {threshold}")
                
                candidates = [p for p in pieces if len(p.split()) >= threshold]
                print(f"         🎯 Candidates meeting threshold: {len(candidates)}")
                
                if candidates:
                    output = max(candidates, key=lambda p: len(p.split()))
                    print(f"         ✅ Selected longest candidate: {len(output.split())} words")
                else:
                    output = max(pieces, key=lambda p: len(p.split()))
                    print(f"         ⚠️  No candidates met threshold, selected longest: {len(output.split())} words")
        
        values.append([output])
        print(f"      ✅ Row {row_idx} output: {len(output)} chars")
    
    # Write to sheet
    range_name = f"{PROCESSED_SHEET_NAME}!{column}2:{column}{len(data) + 1}"
    print(f"\n   📤 Writing to sheet range: {range_name}")
    print(f"   📊 Writing {len(values)} values")
    
    try:
        sheet_mgr.service.spreadsheets().values().update(
            spreadsheetId=spreadsheet_id,
            range=range_name,
            valueInputOption='RAW',
            body={'values': values}
        ).execute()
        print(f"   ✅ Successfully wrote {category} data to sheet")
    except Exception as e:
        print(f"   ❌ Failed to write {category} data: {str(e)}")
        # Don't re-raise, continue processing other categories

# Main function to orchestrate the pipeline
async def main():
    print("🚀 Starting Content Processing Pipeline")
    print("=" * 50)
    
    try:
        print("\n📋 Step 1: Initialize Google Sheets Manager")
        sheet_mgr = GoogleSheetsManager(CREDENTIALS_FILE)
        
        print("\n📋 Step 2: Extract Spreadsheet ID")
        spreadsheet_id = sheet_mgr.extract_spreadsheet_id(SHEET_URL)
        
        print("\n📋 Step 3: Read Data from Sheet")
        data = await read_data_from_sheet(spreadsheet_id, sheet_mgr, CATEGORIES)
        
        print(f"\n📋 Step 4: Process Categories ({len(CATEGORIES)} total)")
        for i, category in enumerate(CATEGORIES, 1):
            print(f"\n🔄 Processing category {i}/{len(CATEGORIES)}: {category}")
            await process_category(data, category, sheet_mgr, spreadsheet_id)
            print(f"✅ Completed category: {category}")
        
        print("\n🎉 Pipeline completed successfully!")
        
    except Exception as e:
        print(f"\n💥 Pipeline failed with error: {str(e)}")
        import traceback
        print("📋 Full traceback:")
        traceback.print_exc()

# Run the pipeline
# if __name__ == "__main__":
#     asyncio.run(main())

print("🔍 Debug version ready - run with: await main()")
await main()

🔍 Debug version ready - run with: await main()
🚀 Starting Content Processing Pipeline

📋 Step 1: Initialize Google Sheets Manager
🔧 Initializing GoogleSheetsManager with credentials: data/url-to-email-445616-cebe4868914f.json
✅ Google Sheets service initialized successfully

📋 Step 2: Extract Spreadsheet ID
🔍 Extracting spreadsheet ID from URL: https://docs.google.com/spreadsheets/d/1wDaFAe5ayIB8zSyjub9QQfsFtyQcHdbgOR4YZmeY8vQ/edit?usp=sharing
✅ Extracted spreadsheet ID: 1wDaFAe5ayIB8zSyjub9QQfsFtyQcHdbgOR4YZmeY8vQ

📋 Step 3: Read Data from Sheet

📖 Reading data from spreadsheet: 1wDaFAe5ayIB8zSyjub9QQfsFtyQcHdbgOR4YZmeY8vQ
   Categories to process: ['ABOUT_US', 'EBOOK', 'COURSES', 'RECENT_BLOG', 'TESTIMONIALS', 'WEBINAR', 'SERVICES', 'PODCAST', 'SHOP']
   Column mapping: {'ABOUT_US': 'M', 'EBOOK': 'N', 'COURSES': 'O', 'RECENT_BLOG': 'P', 'TESTIMONIALS': 'Q', 'WEBINAR': 'R', 'SERVICES': 'S', 'PODCAST': 'T', 'SHOP': 'U'}
   📋 Reading ranges: ['Sheet1!M2:M', 'Sheet1!N2:N', 'Sheet1!O2:O',